# 155 mm Artillery Shell — Range Table

Generates a **range table** in the style of real artillery firing tables, using the full 6-DOF model with **Mach-dependent drag**.

### Parameters
- **Muzzle velocities:** 300 – 900 m/s (step 100)
- **Elevation angles:** 10° – 75° (step 5°)
- **Drag model:** Mach-dependent $C_D(M)$ from McCoy lookup table
- **Aero moments:** Pitch-damping, roll-damping, Magnus moment all active

### Recorded Quantities
| Column | Unit | Description |
|--------|------|-------------|
| Max Range | m | Horizontal distance at ground impact |
| Time of Flight | s | Launch to impact |
| Max Altitude | m | Apogee altitude |
| Impact Velocity | m/s | Speed at ground impact |

In [ ]:
import sys, math, os
import numpy as np
import pandas as pd
sys.path.insert(0, '..')

from src.atmosphere import ISAAtmosphere
from src.aerodynamics import AeroModel
from src.integrator import TrajectoryIntegrator
from src.projectile_config import SHELL_155MM

In [ ]:
atmosphere = ISAAtmosphere()
config = SHELL_155MM

aero = AeroModel(
    Cd=config['Cd'], Cl=config['Cl'], Cm=config['Cm'],
    reference_area_m2=config['reference_area_m2'],
    reference_diameter_m=config['diameter_m'],
    Cmq=config['Cmq'], Clp=config['Clp'], Cnpa=config['Cnpa'],
    mach_table=config['mach_table'],
    cd_table=config['cd_table'],
    reference_length_m=config['length_m'],
)
integrator = TrajectoryIntegrator()

SPIN_RATE_RAD_S = 1800.0
DT_S = 0.05
SIM_TIME_S = 600.0

muzzle_velocities_ms = range(300, 901, 100)
elevation_angles_deg = range(10, 76, 5)
print(f'Generating range table: {len(list(muzzle_velocities_ms))} velocities × {len(list(elevation_angles_deg))} angles')

In [ ]:
records = []
total_runs = len(list(muzzle_velocities_ms)) * len(list(elevation_angles_deg))
run_idx = 0

for v0 in muzzle_velocities_ms:
    for elev_deg in elevation_angles_deg:
        run_idx += 1
        elev_rad = math.radians(elev_deg)
        half = elev_rad / 2.0
        initial_state = np.array([
            0.0, 0.0, 0.0,
            v0 * math.cos(elev_rad), 0.0, v0 * math.sin(elev_rad),
            math.cos(half), 0.0, -math.sin(half), 0.0,
            SPIN_RATE_RAD_S, 0.0, 0.0,
        ])

        result = integrator.integrate(
            initial_state=initial_state,
            t_span=(0.0, SIM_TIME_S),
            dt=DT_S,
            mass_kg=config['mass_kg'],
            inertia_tensor=config['inertia_tensor'],
            aero_model=aero,
            atmosphere=atmosphere,
            wind_vector_ms=np.zeros(3),
        )

        records.append({
            'Muzzle Velocity [m/s]': v0,
            'Elevation [deg]': elev_deg,
            'Max Range [m]': round(result.state[-1, 0], 1),
            'Time of Flight [s]': round(result.time_s[-1], 2),
            'Max Altitude [m]': round(np.max(result.state[:, 2]), 1),
            'Impact Velocity [m/s]': round(result.speed_ms[-1], 1),
        })

        if run_idx % 14 == 0 or run_idx == total_runs:
            print(f'  {run_idx}/{total_runs} runs complete …')

print('\n✅ All runs complete.')

In [ ]:
df = pd.DataFrame(records)

csv_path = os.path.join('..', 'range_table.csv')
df.to_csv(csv_path, index=False)
print(f'Saved range table to {csv_path}  ({len(df)} entries)')

df.head(20)

In [ ]:
# Visualise: Range vs Elevation for a few muzzle velocities
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
for v0 in [300, 500, 700, 900]:
    sub = df[df['Muzzle Velocity [m/s]'] == v0]
    ax.plot(sub['Elevation [deg]'], sub['Max Range [m]'] / 1000,
            marker='o', markersize=4, label=f'{v0} m/s')
ax.set_xlabel('Elevation [°]')
ax.set_ylabel('Range [km]')
ax.set_title('155 mm Shell — Range vs Elevation (Mach-dependent Cd)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()